# Práctica 4. Procesamiento audio
## Ejercicio 1
Construir un identificador de notas musicales. Es decir; en su versión más sencilla  (y  suficiente) la entrada es un sonido con una sola nota musical y debe identificar cuál es. Por simplicidad  elija un único instrumento para la identificación.  


In [ ]:
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
from scipy.io import wavfile

NOTES_SPANISH = {
    'C': 'Do', 'C#': 'Do#', 'Db': 'Reb', 'D': 'Re', 'D#': 'Re#', 'Eb': 'Mib',
    'E': 'Mi', 'F': 'Fa', 'F#': 'Fa#', 'Gb': 'Solb', 'G': 'Sol',
    'G#': 'Sol#', 'Ab': 'Lab', 'A': 'La', 'A#': 'La#', 'Bb': 'Sib', 'B': 'Si'
}

EQUIVALENCIAS = {
    'Do': 'Do', 'Do#': 'Do#/Reb', 'Re': 'Re', 'Re#': 'Re#/Mib',
    'Mi': 'Mi', 'Fa': 'Fa', 'Fa#': 'Fa#/Solb', 'Sol': 'Sol',
    'Sol#': 'Sol#/Lab', 'La': 'La', 'La#': 'La#/Sib', 'Si': 'Si'
}

def identificar_nota(archivo_audio):
    sample_rate, audio_data = wavfile.read(archivo_audio)
    
    if len(audio_data.shape) == 2:
        audio_data = audio_data.mean(axis=1)
    
    audio_data = audio_data.astype(float)
    
    N = len(audio_data)
    yf = fft(audio_data)
    xf = fftfreq(N, 1/sample_rate)
    
    yf_abs = np.abs(yf[:N//2])
    xf_positive = xf[:N//2]
    
    peaks, _ = find_peaks(yf_abs, height=np.max(yf_abs)*0.1)
    
    if len(peaks) > 0:
        fundamental_idx = peaks[np.argmax(yf_abs[peaks])]
        frecuencia_fundamental = abs(xf_positive[fundamental_idx])
    else:
        fundamental_idx = np.argmax(yf_abs)
        frecuencia_fundamental = abs(xf_positive[fundamental_idx])
    
    notas = ['Do', 'Do#', 'Re', 'Re#', 'Mi', 'Fa', 'Fa#', 'Sol', 'Sol#', 'La', 'La#', 'Si']
    
    frecuencias_base = {
        'Do': 16.35, 'Do#': 17.32, 'Re': 18.35, 'Re#': 19.45,
        'Mi': 20.60, 'Fa': 21.83, 'Fa#': 23.12, 'Sol': 24.50,
        'Sol#': 25.96, 'La': 27.50, 'La#': 29.14, 'Si': 30.87
    }
    
    min_diferencia = float('inf')
    nota_detectada = None
    
    for nota in notas:
        freq_base = frecuencias_base[nota]
        freq_nota = freq_base
        
        while freq_nota < 8000:
            diferencia = abs(frecuencia_fundamental - freq_nota)
            if diferencia < min_diferencia:
                min_diferencia = diferencia
                nota_detectada = nota
            freq_nota *= 2
    
    return nota_detectada, frecuencia_fundamental

if __name__ == "__main__":
    for nota_original in ["A", "Ab", "B", "Bb", "C", "D", "Db", "E", "Eb", "F", "G", "Gb"]:
        archivo = f"data/E1/Piano.ff.{nota_original}4.wav"
        nota, frecuencia = identificar_nota(archivo)
        print(f"Nota original : {NOTES_SPANISH[nota_original]}")
        if nota in EQUIVALENCIAS:
            print(f"Nota detectada: {EQUIVALENCIAS[nota]}")
        else:
            print(f"Nota detectada: {nota}")
        print(f"Frecuencia fundamental: {frecuencia:.2f} Hz")
        print()


Nota original : La
Nota detectada: La
Frecuencia fundamental: 440.79 Hz

Nota original : Lab
Nota detectada: Sol#/Lab
Frecuencia fundamental: 416.91 Hz

Nota original : Si
Nota detectada: Si
Frecuencia fundamental: 494.76 Hz

Nota original : Sib
Nota detectada: La#/Sib
Frecuencia fundamental: 467.03 Hz

Nota original : Do
Nota detectada: Do
Frecuencia fundamental: 262.17 Hz

Nota original : Re
Nota detectada: Re
Frecuencia fundamental: 588.99 Hz

Nota original : Reb
Nota detectada: Do#/Reb
Frecuencia fundamental: 556.36 Hz

Nota original : Mi
Nota detectada: Mi
Frecuencia fundamental: 330.87 Hz

Nota original : Mib
Nota detectada: Re#/Mib
Frecuencia fundamental: 624.19 Hz

Nota original : Fa
Nota detectada: Fa
Frecuencia fundamental: 350.37 Hz

Nota original : Sol
Nota detectada: Sol
Frecuencia fundamental: 392.76 Hz

Nota original : Solb
Nota detectada: Fa#/Solb
Frecuencia fundamental: 370.49 Hz




(APORTES ADICIONALES) 

a) El caso más sencillo es el del piano, pero se valorará que se haga con otros instrumentos como la guitarra, la trompeta...  

b) Se valorará que se identifiquen octavas de notas 

c) Identificación de acordes (complejo pero espectacular) 

d) Aportes adicionales